In [1]:
!pip install -U datasets

In [2]:
import json
from transformers import AutoTokenizer
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForTokenClassification
import torch
from torch.optim import AdamW
from transformers import TrainingArguments
from transformers import DataCollatorForTokenClassification

### Loading subset of data

In [3]:
samples = []

with open("train_indicqa.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        samples.append(json.loads(line))
        if i == 31:   # first 32 samples
            break

dataset = Dataset.from_list(samples)

print(dataset)

Dataset({
    features: ['question', 'original_answer', 'tokens', 'labels'],
    num_rows: 32
})


## LOad Tokenizer

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-large")

MAX_LENGTH = 512

## Align Labels

In [5]:
def tokenize_and_align_labels(examples):

    # Pre-tokenize questions into words as expected by is_split_into_words=True
    processed_questions = [q.split() for q in examples["question"]]

    tokenized_inputs = tokenizer(
        processed_questions, # Pass the word-split questions
        examples["tokens"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        padding="max_length",
        is_split_into_words=True,
    )

    all_labels = []

    for i, labels in enumerate(examples["labels"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)
        sequence_ids = tokenized_inputs.sequence_ids(batch_index=i)

        previous_word = None
        label_ids = []

        for word_id, seq_id in zip(word_ids, sequence_ids):

            # Special tokens
            if word_id is None:
                label_ids.append(-100)

            # Question tokens
            elif seq_id == 0:
                label_ids.append(-100)

            # First subword of each context token
            elif word_id != previous_word:
                label_ids.append(labels[word_id])

            # Remaining subwords
            else:
                label_ids.append(-100)

            previous_word = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs

In [6]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

In [7]:
tokenized_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

## Data Loader


In [8]:
train_loader = DataLoader(
    tokenized_dataset,
    batch_size=4,
    shuffle=True
)

In [9]:
model = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-large",
    num_labels=2
)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([4, 512])
torch.Size([4, 512])
torch.Size([4, 512])


In [11]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=10,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    learning_rate=2e-5,
    weight_decay=0.01,

    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="no",

    report_to="none",
)

In [12]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    true_predictions = []

    for prediction, label in zip(predictions, labels):

        for p, l in zip(prediction, label):

            if l != -100:

                true_labels.append(l)
                true_predictions.append(p)

    return {
        "f1": f1_score(true_labels, true_predictions)
    }

In [13]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [14]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics,
)
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.223228,0.205395,0.000000
2,0.201174,0.190972,0.000000
3,0.189282,0.168425,0.000000
4,0.105256,0.126331,0.386555
5,0.105438,0.068689,0.725603
6,0.048635,0.042827,0.869801
7,0.034935,0.034474,0.911025
8,0.025502,0.031139,0.925926
9,0.045150,0.027125,0.929236
10,0.018092,0.025788,0.929104


TrainOutput(global_step=80, training_loss=0.11505175065249204, metrics={'train_runtime': 145.2008, 'train_samples_per_second': 2.204, 'train_steps_per_second': 0.551, 'total_flos': 297186237480960.0, 'train_loss': 0.11505175065249204, 'epoch': 10.0})

In [15]:
pred = trainer.predict(tokenized_dataset)

logits = pred.predictions
labels = pred.label_ids

predictions = np.argmax(logits, axis=-1)

valid_preds = []
valid_labels = []

for p, l in zip(predictions, labels):
    mask = l != -100
    valid_preds.extend(p[mask])
    valid_labels.extend(l[mask])

print("Unique predictions:", np.unique(valid_preds, return_counts=True))
print("Unique labels:", np.unique(valid_labels, return_counts=True))

Unique predictions: (array([0, 1]), array([9381,  534]))
Unique labels: (array([0, 1]), array([9377,  538]))


In [30]:
def predict(sample):
    model.eval()

    # Tokenize question and context
    encoding = tokenizer(
        sample["question"].split(),
        sample["tokens"],
        is_split_into_words=True,
        truncation="only_second",
        max_length=512,
        return_tensors="pt"
    )

    word_ids = encoding.word_ids(batch_index=0)
    sequence_ids = encoding.sequence_ids(batch_index=0)

    encoding = {k: v.to(device) for k, v in encoding.items()}

    # Model prediction
    with torch.no_grad():
        outputs = model(**encoding)

    pred_labels = outputs.logits.argmax(dim=-1).squeeze().cpu().tolist()

    # Convert predictions back to original tokens
    compressed_tokens = []
    previous_word = None

    for pred, word_id, seq_id in zip(pred_labels, word_ids, sequence_ids):

        # Skip special tokens
        if word_id is None:
            continue

        # Skip question tokens
        if seq_id != 1:
            continue

        # Keep only first subword
        if word_id == previous_word:
            continue

        if pred == 1:
            compressed_tokens.append(sample["tokens"][word_id])

        previous_word = word_id

    return " ".join(compressed_tokens)

In [35]:
sample = dataset[0]

print("=" * 100)
print("QUESTION:")
print(sample["question"])

print("\nANSWER:")
print(sample["original_answer"])

print("Original Context")
print(" ".join(sample['tokens'])) # Fixed: call join on the separator string

print("\nMODEL Context:")
print(predict(sample))

QUESTION:
राजधानी दिल्ली में वाहनों की कुल कितनी संख्या है?

ANSWER:
११२ लाख
Original Context
दिल्ली के सार्वजनिक यातायात के साधन मुख्यतः बस, ऑटोरिक्शा और मेट्रो रेल सेवा हैं। दिल्ली की मुख्य यातायात आवश्यकता का ६०% बसें पूरा करती हैं। दिल्ली परिवहन निगम द्वारा संचालित सरकारी बस सेवा दिल्ली की प्रधान बस सेवा है। दिल्ली परिवहन निगम विश्व की सबसे बड़ी पर्यावरण सहयोगी बस-सेवा प्रदान करता है। हाल ही में बी आर टी कॉरिडोर की सेवा अंबेडकर नगर और दिल्ली गेट के बीच आरम्भ हुई है। जिसे तकनीकी खामियों के कारण तोड़ दिया है और अब सड़क को पुराने स्वरूप में बदल दिया गया है। ऑटो रिक्शा दिल्ली में यातायात का एक प्रभावी माध्यम है। ये ईंधन के रूप में सी एन जी का प्रयोग करते हैं, व इनका रंग हरा होता है। दिल्ली में वातानुकूलित टैक्सी सेवा भी उपलब्ध है जिनका किराया ७.५० से १५ रु/कि॰मी॰ तक है। दिल्ली की कुल वाहन संख्या का ३०% निजी वाहन हैं। दिल्ली में १९२२.३२ कि॰मी॰ की लंबाई प्रति १०० कि॰मी॰², के साथ भारत का सर्वाधिक सड़क घनत्व मिलता है। दिल्ली भारत के पांच प्रमुख महानगरों से राष्ट्रीय राजमार्गों द्वारा जुड़ा